# Gen.is.IA Colab Pro ML Training Pipeline

Cloud handoff skeleton for CPU/GPU-heavy ML experiments. The notebook assumes Google Drive root `/content/drive/MyDrive/genisia/` and avoids local-only paths.

## 1. Mount Google Drive and Load Project

In [ ]:
from pathlib import Path
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/genisia')
PROJECT_ROOT = DRIVE_ROOT / 'investment-research-platform-pro' / 'research_platform_definitive'
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
OUTPUT_ROOT = PROJECT_ROOT / 'output'
print('PROJECT_ROOT=', PROJECT_ROOT)

## 2. Feature Engineering: Load Factor Registry and Optional Blocks

In [ ]:
import numpy as np
import pandas as pd
from ml_stock_lab.factor_registry import FACTOR_BLOCKS, feature_columns_for_blocks
from factors.eps_factors import build_eps_factors
from simulation.monte_carlo import monte_carlo_returns

panel_path = OUTPUT_ROOT / 'ml_training_lab' / 'tables' / 'MLTraining_panel.csv'
if panel_path.exists():
    panel = pd.read_csv(panel_path, parse_dates=['date'])
else:
    panel = pd.DataFrame()
print('loaded panel rows:', len(panel))
print('available blocks:', list(FACTOR_BLOCKS))

## 2.1 Auto-Import Canonical Factors and Validate Data

In [ ]:
from ml_stock_lab.factor_registry import FACTOR_REGISTRY

CANONICAL_BLOCKS = [block_id for block_id, block in FACTOR_BLOCKS.items() if not block.experimental]
CANONICAL_FACTOR_SPECS = {name: spec for name, spec in FACTOR_REGISTRY.items() if not spec.get('experimental', True)}
print('canonical blocks:', CANONICAL_BLOCKS)
print('canonical factor specs:', list(CANONICAL_FACTOR_SPECS))

def validate_training_matrix(df, max_nan_share=0.30):
    if df.empty:
        return pd.DataFrame(columns=['feature', 'nan_share', 'status'])
    numeric = df.select_dtypes(include='number')
    report = numeric.isna().mean().rename('nan_share').reset_index().rename(columns={'index': 'feature'})
    report['status'] = np.where(report['nan_share'] > max_nan_share, 'REVIEW', 'OK')
    bad = report[report['status'].eq('REVIEW')]
    assert bad.empty, f"Columns with NaN share > {max_nan_share:.0%}: {bad['feature'].tolist()[:20]}"
    return report

# Run after selecting model features: validate_training_matrix(panel[features])

## 3. Model Training: LightGBM + Ridge + Optional LSTM

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
try:
    import lightgbm as lgb
except Exception:
    lgb = None

TARGET = 'forward_return_21d'
FEATURE_BLOCKS = ['value', 'quality', 'momentum', 'risk', 'size', 'growth', 'eps_factors', 'macro_context']

def fit_models(train_df, test_df):
    features = feature_columns_for_blocks(train_df, FEATURE_BLOCKS, target=TARGET, min_non_null=20)
    X_train = train_df[features].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    y_train = pd.to_numeric(train_df[TARGET], errors='coerce').fillna(0.0)
    X_test = test_df[features].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    models = {'ridge': Ridge(alpha=1.0).fit(X_train, y_train)}
    if lgb is not None:
        models['lightgbm'] = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42).fit(X_train, y_train)
    preds = {name: model.predict(X_test) for name, model in models.items()}
    return models, preds, features

# Optional GPU LSTM placeholder: add PyTorch sequence dataset only after panel coverage is confirmed.

## 4. Walk-Forward Cross-Validation (No Leakage)

In [ ]:
def expanding_walk_forward(panel, start_test='2019-01-01', step_months=6):
    if panel.empty or TARGET not in panel.columns or 'date' not in panel.columns:
        return []
    panel = panel.sort_values('date')
    cutoffs = pd.date_range(start_test, panel['date'].max(), freq=f'{step_months}MS')
    folds = []
    for cutoff in cutoffs:
        train = panel[panel['date'] < cutoff]
        test = panel[(panel['date'] >= cutoff) & (panel['date'] < cutoff + pd.DateOffset(months=step_months))]
        if len(train) > 1000 and len(test) > 100:
            folds.append((cutoff, train, test))
    return folds

folds = expanding_walk_forward(panel)
print('folds:', len(folds))

## 5. SHAP Feature Importance per Factor Group

In [ ]:
try:
    import shap
except Exception:
    shap = None

def shap_summary(model, X):
    if shap is None:
        return pd.DataFrame()
    explainer = shap.Explainer(model, X)
    values = explainer(X)
    return pd.DataFrame({'feature': X.columns, 'mean_abs_shap': np.abs(values.values).mean(axis=0)})

## 6. Factor IC Analysis

In [ ]:
def rank_ic_by_date(df, score_col, target_col=TARGET):
    rows = []
    for date, group in df.groupby('date'):
        if group[score_col].notna().sum() >= 5 and group[target_col].notna().sum() >= 5:
            rows.append({'date': date, 'rank_ic': group[score_col].rank().corr(group[target_col].rank())})
    return pd.DataFrame(rows)

# Example after predictions: rank_ic_by_date(prediction_frame, 'score_lightgbm')

## 6.1 Save IC Heatmap

In [ ]:
def save_ic_heatmap(ic_frame, output_path):
    import matplotlib.pyplot as plt
    if ic_frame is None or ic_frame.empty:
        return None
    pivot = ic_frame.copy()
    if {'feature_group', 'horizon', 'ic'}.issubset(pivot.columns):
        pivot = pivot.pivot_table(index='feature_group', columns='horizon', values='ic', aggfunc='mean')
    numeric = pivot.select_dtypes(include='number') if isinstance(pivot, pd.DataFrame) else pd.DataFrame()
    if numeric.empty:
        return None
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(numeric.values, aspect='auto', cmap='RdYlGn', vmin=-0.1, vmax=0.1)
    ax.set_xticks(range(numeric.shape[1]), labels=[str(c) for c in numeric.columns], rotation=45, ha='right')
    ax.set_yticks(range(numeric.shape[0]), labels=[str(i) for i in numeric.index])
    fig.colorbar(im, ax=ax, label='IC')
    fig.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=160)
    plt.close(fig)
    return output_path

# Example: save_ic_heatmap(ic_summary, EXPORT_ROOT / 'ic_heatmap.png')

## 7. Export Artifacts to Drive

In [ ]:
import json
from datetime import datetime, timezone
EXPORT_ROOT = DRIVE_ROOT / 'ResearchPlatformArtifacts' / 'ml_training_colab'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

metrics = {'status': 'SKELETON_READY', 'feature_blocks': FEATURE_BLOCKS, 'target': TARGET}
(EXPORT_ROOT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

runs_log_path = EXPORT_ROOT / 'runs_log.json'
if runs_log_path.exists():
    runs_log = json.loads(runs_log_path.read_text(encoding='utf-8'))
else:
    runs_log = []
runs_log.append({
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'n_factors': int(len(FEATURE_BLOCKS)),
    'best_model': metrics.get('best_model', 'TBD'),
    'status': metrics['status'],
})
runs_log_path.write_text(json.dumps(runs_log, indent=2), encoding='utf-8')
print('export root:', EXPORT_ROOT)